In [ ]:
import torch
import torch.nn as nn
from datasets import load_dataset

# 注意：请先从 ModelScope 下载数据集
# modelscope download --dataset modelscope/mnist --local_dir ./mnist_data

# 数据加载
dataset = load_dataset(
    "parquet", 
    data_files="./mnist_data/mnist/train-00000-of-00001.parquet",
    split="train"
)

print(f"数据集已加载，共 {len(dataset)} 条样本。")
print(f"列名: {dataset.column_names}")

# 数据预处理
def preprocess(dataset):
    processed_images = []
    for img in dataset['image']:
        pixel_data = list(img.getdata())
        img_tensor = torch.tensor(pixel_data, dtype=torch.float32)
        normalized_img = img_tensor / 255.0
        processed_images.append(normalized_img)
    images = torch.stack(processed_images)
    labels = torch.tensor(dataset['label'], dtype=torch.long)
    return images, labels

images, labels = preprocess(dataset)

model = nn.Sequential(nn.Flatten(), # 将 28x28 图像展平为 784 维向量
                      nn.Linear(784, 512),  # 输入 784 维，输出 512 维
                      nn.ReLU(),  # 激活函数
                      nn.Linear(512, 256),  # 输入 512 维，输出 256 维
                      nn.ReLU(),  # 激活函数
                      nn.Linear(256,  10)   # 输入 256 维，输出 10 维
                     )

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

数据集已加载，共 60000 条样本。
列名: ['image', 'label']


In [2]:
# 这 3 行是把数据移动到 GPU 上，如果没有 GPU 不要加这 3 行
images = images.to('cuda')
labels = labels.to('cuda')
model = model.to('cuda')

for epoch in range(2000):
    optimizer.zero_grad()

    # 前向传播
    preds = model(images)  # 输出形状: [N, 10]

    # 计算损失
    loss = loss_fn(preds, labels)

    # 反向传播
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

Epoch 0, Loss: 2.3070
Epoch 10, Loss: 2.2660
Epoch 20, Loss: 2.2094
Epoch 30, Loss: 2.1117
Epoch 40, Loss: 1.9378
Epoch 50, Loss: 1.6640
Epoch 60, Loss: 1.3382
Epoch 70, Loss: 1.0658
Epoch 80, Loss: 0.8806
Epoch 90, Loss: 0.7574
Epoch 100, Loss: 0.6718
Epoch 110, Loss: 0.6094
Epoch 120, Loss: 0.5623
Epoch 130, Loss: 0.5257
Epoch 140, Loss: 0.4965
Epoch 150, Loss: 0.4728
Epoch 160, Loss: 0.4533
Epoch 170, Loss: 0.4369
Epoch 180, Loss: 0.4230
Epoch 190, Loss: 0.4110
Epoch 200, Loss: 0.4005
Epoch 210, Loss: 0.3913
Epoch 220, Loss: 0.3831
Epoch 230, Loss: 0.3757
Epoch 240, Loss: 0.3690
Epoch 250, Loss: 0.3629
Epoch 260, Loss: 0.3573
Epoch 270, Loss: 0.3520
Epoch 280, Loss: 0.3471
Epoch 290, Loss: 0.3426
Epoch 300, Loss: 0.3382
Epoch 310, Loss: 0.3341
Epoch 320, Loss: 0.3302
Epoch 330, Loss: 0.3265
Epoch 340, Loss: 0.3229
Epoch 350, Loss: 0.3195
Epoch 360, Loss: 0.3162
Epoch 370, Loss: 0.3131
Epoch 380, Loss: 0.3100
Epoch 390, Loss: 0.3070
Epoch 400, Loss: 0.3041
Epoch 410, Loss: 0.3013
Epo

In [4]:
def predict(pil_img):
    with torch.no_grad():
        pixel_data = list(pil_img.getdata())
        img_tensor = torch.tensor(pixel_data, dtype=torch.float32)
        normalized_img = img_tensor / 255.0
        input_batch = normalized_img.unsqueeze(0).to('cuda')  # 这里把测试数据也放到 GPU 中

        outputs = model(input_batch)  # [1, 10]
        _, predicted = torch.max(outputs, 1)
        return predicted.item()

test_dataset = load_dataset(
    "parquet",
    data_files="./mnist_data/mnist/test-00000-of-00001.parquet",
    split="train"
)

# 随机测试 10 个样本
import random
for idx in random.sample(range(len(test_dataset)), 10):
    test_img = test_dataset[idx]["image"]
    true_label = test_dataset[idx]["label"]
    predicted_label = predict(test_img)

    status = "right" if (true_label == predicted_label) else "wrong"
    print(f"索引 {idx:3d} | 真实: {true_label} | 预测: {predicted_label} | {status}")

索引 6567 | 真实: 0 | 预测: 0 | right
索引 8809 | 真实: 1 | 预测: 1 | right
索引 5358 | 真实: 3 | 预测: 3 | right
索引 869 | 真实: 5 | 预测: 5 | right
索引 8682 | 真实: 1 | 预测: 1 | right
索引 8360 | 真实: 1 | 预测: 1 | right
索引 8949 | 真实: 9 | 预测: 9 | right
索引 3887 | 真实: 8 | 预测: 8 | right
索引 1771 | 真实: 2 | 预测: 2 | right
索引 3193 | 真实: 3 | 预测: 3 | right
